# LoRA training: `s1_own` (4x L4)
Generated by `kaggle/train/make_train_notebook.py` from `kaggle/train/trainkit/` (edit there, not here).
Recipe = training probe v4: NF4 from the bf16 QAT weights, LoRA r16 on 410 linears, FSDP over 4 ranks, ≤8k windows,
loss on model-written tokens only. Data: `train_s1_own.npz`.

## 1. Settings
Everything the run needs is in `train_cfg.json`; the budget clock starts here.

In [ ]:
import json, os, sys, time
for d in ("/kaggle/working/evalkit", "/kaggle/working/trainkit"):
    os.makedirs(d, exist_ok=True)
TRAIN_CFG = {'name': 's1_own', 'lr': 5e-05, 'max_windows': None, 'budget_h': 3.0, 'smoke': False, 'epochs': 2.0, 'grad_accum': 2, 'save_every': 9, 'val_every': 0}
TRAIN_CFG["t0"] = time.time()
open("/kaggle/working/train_cfg.json", "w").write(json.dumps(TRAIN_CFG, indent=1))
TRAIN_CFG

## 2. Embedded inputs
Tokenized windows (unless attached as a dataset) and, for the smoke model, the real config.

In [ ]:
import base64, pathlib
DATA_B64 = '...'
if DATA_B64:
    pathlib.Path("/kaggle/working/train_data.npz").write_bytes(base64.b64decode(DATA_B64))
G4CFG = ''
if G4CFG:
    pathlib.Path("/kaggle/working/g4cfg.json").write_text(G4CFG)
print("data embedded:", bool(DATA_B64), "| smoke config:", bool(G4CFG))

## 3. Environment modules (from evalkit)

### `evalkit/__init__.py`
evalkit: scorer-faithful local evaluation of a submission on Kaggle 4x L4 (harness swegemma 0.2.11 / adk-submission 0.2.13).

In [ ]:
%%writefile /kaggle/working/evalkit/__init__.py
"""
evalkit: scorer-faithful local evaluation of a submission on Kaggle 4x L4 (harness swegemma 0.2.11 / adk-submission 0.2.13).

Modules, in the order the notebook uses them:
  config      Settings (one JSON file shared by every step) and fixed paths
  util        run.log logger + shell helper
  envsetup    ScorerEnv: restore the Py3.12 scorer venv, upgrade changed wheelhouse wheels, add test deps (stdlib only:
              runs in the notebook's own Python 3.13 kernel)
  inputs      Inputs: competition / dev-set tasks, snapshots, served model
  submission  Submission: unpack the submission under test, declared model, adapters, eval budget
  server      ScorerServer: vLLM with the scorer's settings + model registry; offline sandbox
  runner      stratified task order + ChunkedEvaluation (official Evaluator, sequential, rep-major repeats)
  report      Report: merge chunks per rep, pass@k summary, trace analysis, package
  main        orchestration of everything after the env step (runs inside the venv)
Source of truth: kaggle/eval_l4/evalkit/. make_eval_notebook.py turns it into the competition notebook.
"""

### `evalkit/config.py`
Run settings and fixed paths. Stdlib only, so the env step can import it outside the venv.

In [ ]:
%%writefile /kaggle/working/evalkit/config.py
"""Run settings and fixed paths. Stdlib only, so the env step can import it outside the venv."""
import json
import pathlib
import time
from dataclasses import asdict, dataclass, field

WORK = pathlib.Path("/kaggle/working")
INPUT = pathlib.Path("/kaggle/input")
VENV = pathlib.Path("/tmp/scorer_host")
TARGET_MODEL = "gemma-4-31b-it-qat-w4a16-ct"
CFG_FILE = WORK / "eval_cfg.json"
SUBMISSION_ZIP = WORK / "submission_src.zip"
ANALYZER = WORK / "trace_analyzer.py"
SANDBOX_ENV = WORK / "sandbox_env.json"   # task-sandbox interpreter + site, written by envsetup
SCORER_SITE = pathlib.Path("/tmp/scorer_site")
SANDBOX_PIP = pathlib.Path("/tmp/sandbox_pip")
VALID_TASKS = WORK / "valid_tasks.json"       # from the preflight sweep; embedded by make_eval_notebook.py


@dataclass
class Settings:
    name: str = "v7_noedit"           # submissions/<name>; label of every output
    task_ids: list | None = None      # None -> all tasks, interleaved by repo (any prefix is a stratified sample)
    max_tasks: int | None = None      # cap the task list (shakedown runs)
    seed: int = 0
    chunk_size: int = 8
    session_budget_h: float = 10.5    # stop scheduling chunks after this; the session cap is 12 h
    est_task_min: float = 5.5         # 4.5 min agent budget + setup/verification
    repeats: int = 1                  # >1: run the list again (rep-major) -> pass@k, trajectories for rejection sampling
    devset: bool = False              # True -> renamed dev set (kernel output gemma4-devset-build)
    valid_only: bool = False          # True -> only tasks in valid_tasks.json (gold passes, empty fails; preflight sweep)
    sweep: bool = False               # preflight: gold + empty over ALL tasks -> valid_tasks.json (else 7-task smoke)
    gate_adapters: list | None = None  # behaviour gate: adapter names to compare with base (evalkit/gate.py)
    gate_samples: int = 4             # behaviour gate: samples per decision point
    t0: float = field(default_factory=time.time)  # notebook start; budget counts from here across processes

    @property
    def label(self):
        return f"eval_{self.name}" + ("_dev" if self.devset else "")

    def hours_left(self):
        return self.session_budget_h - (time.time() - self.t0) / 3600

    def save(self, path=CFG_FILE):
        path.write_text(json.dumps(asdict(self), indent=1))

    @classmethod
    def load(cls, path=CFG_FILE):
        return cls(**json.loads(path.read_text()))

### `evalkit/util.py`
Logger that writes to stdout and /kaggle/working/run.log, plus a logged shell helper. Stdlib only.

In [ ]:
%%writefile /kaggle/working/evalkit/util.py
"""Logger that writes to stdout and /kaggle/working/run.log, plus a logged shell helper. Stdlib only."""
import glob
import subprocess
import time

from .config import WORK


class Log:
    def __init__(self, path=WORK / "run.log"):
        self.f = open(path, "a", buffering=1)

    def __call__(self, *a):
        s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
        print(s, flush=True)
        self.f.write(s + "\n")

    def sh(self, cmd, timeout=None, check=False):
        self("$", cmd[:220])
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = ((r.stdout or "")[-1500:] + ("\n" + r.stderr[-1500:] if (r.stderr or "").strip() else "")).strip()
        if out:
            self(out)
        if check and r.returncode != 0:
            raise SystemExit(f"FAILED rc={r.returncode}: {cmd[:120]}")
        return r


log = Log()


def first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

### `evalkit/envsetup.py`
ScorerEnv: the scorer's Python environment, offline. Runs in the notebook kernel (Python 3.13), stdlib only.

In [ ]:
%%writefile /kaggle/working/evalkit/envsetup.py
"""
ScorerEnv: the scorer's Python environment, offline. Runs in the notebook kernel (Python 3.13), stdlib only.

The wheelhouse is cp312-only and not a consistent dependency set, so the venv is built once on a CPU kernel with
Internet (kaggle/env/build_scorer_env.py -> scorer_env.tgz). Here it is restored, then every wheelhouse wheel whose
version differs from the venv (Oct 9: swegemma 0.2.11, adk-submission 0.2.13; their requirements are unchanged)
is installed with --no-deps, plus the test-only deps of the fastapi tasks (dataset test-deps-wheels).
"""
import json
import os
import pathlib
import re
import sys
import time

from .config import INPUT, SANDBOX_ENV, SANDBOX_PIP, SCORER_SITE, VENV
from .util import first, log

TEST_DEPS = re.compile(r"(inline_snapshot|dirty_equals|asttokens|executing)-")


def canon(name):
    return re.sub(r"[-_.]+", "-", name).lower()


class ScorerEnv:
    def __init__(self, venv=VENV, n_gpus=4):
        self.venv, self.n_gpus = pathlib.Path(venv), n_gpus
        self.python = self.venv / "bin" / "python"

    def check_gpus(self):
        log.sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2")
        gpus = [g for g in log.sh("nvidia-smi --query-gpu=name --format=csv,noheader").stdout.split("\n") if g.strip()]
        if len(gpus) < self.n_gpus:  # don't burn quota on the wrong machine
            raise SystemExit(f"need GPU L4 x{self.n_gpus}, got {gpus}")

    def restore(self):
        tgz = first(f"{INPUT}/**/scorer_env.tgz")
        assert tgz, "attach the output of kernel gemma4-scorer-env (kernel_sources)"
        log.sh(f"tar -xzf {tgz} -C /", timeout=1800, check=True)

    def installed(self):
        out = log.sh(f"{self.python} -c 'import importlib.metadata as m, json; "
                     "print(json.dumps({d.metadata[\"Name\"]: d.version for d in m.distributions()}))'",
                     check=True).stdout.strip().splitlines()[-1]
        return {canon(k): v.split("+")[0] for k, v in json.loads(out).items()}

    def stale_wheels(self):
        swegemma = first(f"{INPUT}/**/swegemma-*.whl")
        assert swegemma, "attach dataset metric/gemma-4-developer-agent-wheelhouse"
        have, stale = self.installed(), []
        for w in sorted(pathlib.Path(swegemma).parent.glob("*.whl")):
            if "cutlass" in w.name.lower():
                continue
            n, v = w.name.split("-")[:2]
            v = re.sub(r"\+?cu\d+$", "", v)
            if have.get(canon(n)) != v:
                log(f"wheelhouse {n} {v} != venv {have.get(canon(n))}")
                stale.append(w)
        return stale

    def test_dep_wheels(self):
        found = [pathlib.Path(p) for p in sorted(map(str, INPUT.rglob("*.whl"))) if TEST_DEPS.match(pathlib.Path(p).name)]
        if not found:
            log("WARNING: dataset test-deps-wheels not attached -> ~32 fastapi tasks will be unverifiable")
        return found

    def install(self, wheels):
        stage = pathlib.Path("/tmp/wh_stage")
        stage.mkdir(exist_ok=True)
        for w in wheels:  # pip rejects 'cu128' without '+' in a wheel's version
            name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
            if not (stage / name).exists():
                os.symlink(w, stage / name)
        whls = sorted(str(p) for p in stage.glob("*.whl"))
        if not whls:
            return
        r = log.sh(f"{sys.executable} -m pip --python {self.python} install --no-deps --no-index -q " + " ".join(whls),
                   timeout=1800)
        if r.returncode != 0:  # fallback: wheels are zip archives, unpack into site-packages
            site = log.sh(f"{self.python} -c 'import site; print(site.getsitepackages()[0])'").stdout.strip()
            for w in whls:
                log.sh(f"cd {site} && unzip -oq {w}", check=True)

    def ensure_pip(self):
        """uv venvs have no pip; the sandbox's per-task venvs need `python -m pip` (found via _host_env.pth) to
        install task packages. pip is pure Python, so copy the notebook Python's own pip into the venv (offline)."""
        if log.sh(f"{self.python} -m pip --version").returncode == 0:
            return
        import pip
        src = pathlib.Path(pip.__file__).parent
        site = pathlib.Path(log.sh(f"{self.python} -c 'import site; print(site.getsitepackages()[0])'",
                                   check=True).stdout.strip())
        log.sh(f"cp -r {src} {site}/ && cp -r {src.parent}/pip-*.dist-info {site}/", check=True)
        log.sh(f"{self.python} -m pip --version", check=True)

    def task_sandbox(self):
        """Task sandboxes run on this notebook's Python 3.13 (the scorer's sandbox is python:3.13-slim) with the
        scorer site built by gemma4-devset-build; pip is copied into its own dir (no ensurepip offline)."""
        import pip
        tgz = first(f"{INPUT}/**/scorer_site.tgz")
        assert tgz, "attach the output of kernel gemma4-devset-build (scorer_site.tgz)"
        SCORER_SITE.mkdir(parents=True, exist_ok=True)
        log.sh(f"tar -xzf {tgz} -C {SCORER_SITE}", timeout=1800, check=True)
        SANDBOX_PIP.mkdir(parents=True, exist_ok=True)
        src = pathlib.Path(pip.__file__).parent
        log.sh(f"cp -r {src} {SANDBOX_PIP}/ && cp -r {src.parent}/pip-*.dist-info {SANDBOX_PIP}/", check=True)
        py = os.path.realpath(sys.executable)
        env = {"python": py, "site": str(SCORER_SITE), "pip": str(SANDBOX_PIP)}
        pins = first(f"{INPUT}/**/overlays/pins.json")  # optional: kernel output gemma4-test-overlays
        if pins:
            root = pathlib.Path(pins).parent
            env.update(extras=str(root / "extras"), pins_root=str(root / "pins"), pins=json.loads(pathlib.Path(pins).read_text()))
            log(f"overlays: extras + pins for {len(env['pins'])} tasks from {root}")
        SANDBOX_ENV.write_text(json.dumps(env))
        sp = f"/tmp/sbx_check/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
        log.sh(f"{py} -m venv --without-pip --clear /tmp/sbx_check && printf '%s\\n%s\\n' {SCORER_SITE} {SANDBOX_PIP} "
               f"> {sp}/_s.pth && /tmp/sbx_check/bin/python3 -c 'import sys, pytest, pip; print(\"task sandbox\", "
               "sys.version.split()[0], \"pytest\", pytest.__version__, \"pip\", pip.__version__)'", check=True)

    def verify(self):
        log.sh(f"{self.python} -c 'import sys, torch, vllm, swegemma, adk_submission, google.adk, "
               "importlib.metadata as m; print(sys.version.split()[0], \"torch\", torch.__version__, \"vllm\", "
               "vllm.__version__, \"swegemma\", m.version(\"swegemma\"), \"adk-submission\", "
               "adk_submission.__version__, \"adk\", google.adk.__version__)'", check=True)
        # import needs pytest's pluggy, which the sandbox venv installs per task; here only check presence
        log.sh(f"{self.python} -c 'import importlib.util as u; "
               "print(\"test deps\", {m: bool(u.find_spec(m)) for m in (\"inline_snapshot\", \"dirty_equals\")})'")

    def prepare_training(self, gpus=True):
        """Scorer venv + the training wheels (peft 0.21.2, bitsandbytes 0.50.2: kernel output gemma4-train-wheels)."""
        t = time.time()
        if gpus:
            self.check_gpus()
        self.restore()
        train = [pathlib.Path(p) for p in sorted(map(str, INPUT.rglob("train_wheels/*.whl")))]
        assert train, "attach the output of kernel gemma4-train-wheels (kernel_sources)"
        self.install(self.stale_wheels() + train)
        self.ensure_pip()
        log.sh(f"{self.python} -c 'import torch, transformers, peft, bitsandbytes; print(\"torch\", torch.__version__, "
               "\"transformers\", transformers.__version__, \"peft\", peft.__version__, \"bnb\", bitsandbytes.__version__)'",
               check=True)
        log(f"training venv ready in {(time.time() - t) / 60:.1f} min")

    def prepare(self, gpus=True):
        t = time.time()
        if gpus:
            self.check_gpus()
        self.restore()
        self.install(self.stale_wheels() + self.test_dep_wheels())
        self.ensure_pip()
        self.task_sandbox()
        self.verify()
        log(f"scorer venv ready in {(time.time() - t) / 60:.1f} min")

## 4. trainkit modules
One cell per module.

### `trainkit/__init__.py`
trainkit: LoRA SFT of gemma-4-31b-it on Kaggle 4x L4 with the recipe measured by the training probe (v4, Oct 10):

In [ ]:
%%writefile /kaggle/working/trainkit/__init__.py
"""
trainkit: LoRA SFT of gemma-4-31b-it on Kaggle 4x L4 with the recipe measured by the training probe (v4, Oct 10):
NF4 from the bf16 QAT weights, LoRA r16 on the 410 language-model linears, FSDP FULL_SHARD over 4 ranks,
memory-efficient SDPA (head_dim 512), loss inside the FSDP root, activation offload for long windows, per-layer export.

Modules, in the order the notebook uses them:
  config    TrainSettings (one JSON shared by every process) and fixed paths
  data      TrainData: the tokenized windows (.npz from posttrain/tokenize_windows.py), val split, per-rank schedule
  model     load NF4 + LoRA + FSDP; MaskedLoss (trained tokens only, Gemma softcap, chunked)
  export    per-layer gather of the LoRA weights -> PEFT adapter that vLLM serves
  worker    the torchrun entry point: train loop, validation loss, periodic + final export
  launcher  runs torchrun inside the scorer venv, keeps per-rank logs
  main      orchestration after the env step (runs inside the venv)
smoke=True runs the same code on CPU (gloo, 2 ranks) with a tiny random Gemma-4 built from the real config.
Source of truth: kaggle/train/trainkit/; make_train_notebook.py turns it into a notebook.
"""

### `trainkit/config.py`
Training settings and fixed paths. Stdlib only.

In [ ]:
%%writefile /kaggle/working/trainkit/config.py
"""Training settings and fixed paths. Stdlib only."""
import json
import pathlib
import time
from dataclasses import asdict, dataclass, field

WORK = pathlib.Path("/kaggle/working")
INPUT = pathlib.Path("/kaggle/input")
CFG_FILE = WORK / "train_cfg.json"
DATA_FILE = WORK / "train_data.npz"       # written by the notebook's inputs cell (or found under /kaggle/input)
OUT_DIR = WORK / "train_out"              # adapters, metrics, per-rank logs
SMOKE_CONFIG = WORK / "g4cfg.json"        # real Gemma-4 config, for the tiny smoke model
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


@dataclass
class TrainSettings:
    name: str = "conv_lora"           # adapter name (adapters/<name>/ in the submission)
    rank: int = 16
    alpha: int = 32
    lr: float = 1e-4
    warmup_steps: int = 5
    grad_accum: int = 2               # optimizer step every grad_accum windows per rank (batch = 4 ranks x accum)
    epochs: float = 1.0
    max_windows: int | None = None    # cap (pipeline smoke runs)
    offload_from: int = 4097          # windows longer than this keep checkpoint activations in host RAM
    val_every: int = 0                # optimizer steps between validation passes (0 = only at the end)
    save_every: int = 0               # optimizer steps between adapter exports (0 = only at the end)
    budget_h: float = 10.5            # stop taking steps after this, then export (session cap 12 h)
    seed: int = 0
    smoke: bool = False               # CPU, gloo, 2 ranks, tiny random model
    t0: float = field(default_factory=time.time)

    def hours_left(self):
        return self.budget_h - (time.time() - self.t0) / 3600

    def save(self, path=CFG_FILE):
        path.write_text(json.dumps(asdict(self), indent=1))

    @classmethod
    def load(cls, path=CFG_FILE):
        return cls(**json.loads(path.read_text()))

### `trainkit/data.py`
TrainData: tokenized windows (posttrain/tokenize_windows.py) -> per-rank schedules of (input_ids, labels).

In [ ]:
%%writefile /kaggle/working/trainkit/data.py
"""TrainData: tokenized windows (posttrain/tokenize_windows.py) -> per-rank schedules of (input_ids, labels)."""
import random

import numpy as np
import torch

IGNORE = -100


class TrainData:
    def __init__(self, path, max_windows=None, vocab=None, seed=0):
        d = np.load(path)
        self.ids, self.train, self.starts = d["ids"], d["train"], d["starts"]
        self.task, self.val = d["task"], d["val"].astype(bool)
        n = len(self.starts) - 1
        w = list(range(n))
        random.Random(seed).shuffle(w)
        self.train_w = [i for i in w if not self.val[i]][:max_windows]
        self.val_w = [i for i in w if self.val[i]]
        self.vocab = vocab  # smoke: fold ids into the tiny model's vocabulary

    def length(self, w):
        return int(self.starts[w + 1] - self.starts[w])

    def example(self, w, device):
        """-> (ids [1, T], labels [T-1]): labels[t] = ids[t+1] if that token is model-written, else IGNORE."""
        s, e = self.starts[w], self.starts[w + 1]
        ids = torch.from_numpy(self.ids[s:e].astype(np.int64))
        if self.vocab:
            ids = ids % self.vocab
        lab = ids[1:].clone()
        lab[torch.from_numpy(self.train[s + 1:e] == 0)] = IGNORE
        return ids.unsqueeze(0).to(device), lab.to(device)

    def schedule(self, windows, rank, world, epochs=1.0):
        """Equal-length per-rank lists (every rank must run the same number of steps under FSDP). Windows are dealt
        in length order so the ranks of one step get similar lengths (no rank waits long for another)."""
        n_total = int(len(windows) * epochs)
        reps = [windows[i % len(windows)] for i in range(n_total)]
        per_step = []
        for i in range(0, len(reps) - world + 1, world):
            group = sorted(reps[i:i + world], key=self.length)
            per_step.append(group)
        return [g[rank] for g in per_step]

    def stats(self):
        lens = [self.length(w) for w in self.train_w]
        return {"train_windows": len(self.train_w), "val_windows": len(self.val_w),
                "train_tokens": int(sum(lens)), "max_len": max(lens) if lens else 0}

### `trainkit/model.py`
Model: NF4 Gemma-4-31B + LoRA r16 + FSDP, and the masked, chunked, soft-capped loss (probe v4 recipe).

In [ ]:
%%writefile /kaggle/working/trainkit/model.py
"""Model: NF4 Gemma-4-31B + LoRA r16 + FSDP, and the masked, chunked, soft-capped loss (probe v4 recipe)."""
import re
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
from torch.distributed.fsdp.wrap import lambda_auto_wrap_policy

from .config import TARGETS
from .data import IGNORE

LORA_RE = r".*language_model.*\.(" + "|".join(TARGETS) + r")$"
CHUNK = 256  # loss chunk: 256 x 262k fp32 logits = 0.27 GB


def load_base(path, rank, smoke):
    """Every rank quantises the bf16 QAT weights to NF4 on its own GPU (~11 min); FSDP then shards them."""
    from transformers import AutoModelForImageTextToText
    if smoke:
        return AutoModelForImageTextToText.from_pretrained(path, dtype=torch.float32, attn_implementation="sdpa")
    from transformers import BitsAndBytesConfig
    q = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,  # FSDP can only shard 4-bit weights stored in the compute dtype
        llm_int8_skip_modules=["vision_tower", "embed_vision", "multi_modal_projector", "audio_tower", "embed_audio",
                               "lm_head"])
    return AutoModelForImageTextToText.from_pretrained(path, dtype=torch.bfloat16, attn_implementation="sdpa",
                                                       quantization_config=q, device_map={"": rank})


def add_lora(m, r, alpha, dtype):
    from peft import LoraConfig, get_peft_model
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    m.enable_input_require_grads()
    m = get_peft_model(m, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.0, bias="none", target_modules=LORA_RE,
                                     task_type="CAUSAL_LM"))
    for n, p in m.named_parameters():
        if "lora_" in n:
            p.data = p.data.to(dtype)  # FSDP flat params need one dtype per unit
    return m


class MaskedCE(torch.autograd.Function):
    """Next-token CE over the trained positions only, with Gemma's final-logit softcap, CHUNK positions at a time.
    The head is frozen, so only dL/dh is needed; it is computed in forward and kept (T x hidden)."""

    @staticmethod
    def forward(ctx, h, w, y, cap):
        idx = (y != IGNORE).nonzero().squeeze(1)
        gh = torch.zeros_like(h)
        tot = torch.zeros((), device=h.device, dtype=torch.float32)
        n = max(1, idx.numel())
        for i in range(0, idx.numel(), CHUNK):
            j = idx[i:i + CHUNK]
            hc = h[j].detach().requires_grad_(True)
            with torch.enable_grad():
                z = (hc @ w.T).float()
                if cap:
                    z = torch.tanh(z / cap) * cap
                loss = F.cross_entropy(z, y[j], reduction="sum")
                g, = torch.autograd.grad(loss, hc)
            gh[j] = g
            tot += loss.detach()
        ctx.save_for_backward(gh)
        ctx.n = n
        return tot / n

    @staticmethod
    def backward(ctx, go):
        gh, = ctx.saved_tensors
        return gh * (go / ctx.n), None, None, None


class LossWrap(nn.Module):
    """FSDP root: the tied lm_head/embedding is unsharded only inside the root forward, so the loss lives here."""

    def __init__(self, m, cap):
        super().__init__()
        self.m, self.cap = m, cap

    def forward(self, ids, labels):
        base = self.m.base_model.model  # PeftModel -> LoraModel -> Gemma4ForConditionalGeneration
        h = base.model.language_model(input_ids=ids, use_cache=False).last_hidden_state[0]
        return MaskedCE.apply(h[:-1], base.get_output_embeddings().weight, labels, self.cap)


def wrap_fsdp(m, smoke):
    """One FSDP unit per decoder layer, the vision tower in its own (never gathered) unit, the rest in the root."""
    lm = next(mm for n, mm in m.named_modules() if n.endswith("language_model") and "vision" not in n)
    layer_cls = type(lm.layers[0])
    vision = [mm for n, mm in m.named_modules() if n.endswith("vision_tower") or n.endswith("embed_vision")]
    types = tuple({layer_cls} | {type(v) for v in vision})
    cap = getattr(m.config.get_text_config(), "final_logit_softcapping", None)
    fm = FSDP(LossWrap(m, cap), auto_wrap_policy=partial(lambda_auto_wrap_policy, lambda_fn=lambda x: isinstance(x, types)),
              sharding_strategy=ShardingStrategy.FULL_SHARD, use_orig_params=True,
              device_id=torch.device("cpu") if smoke else torch.cuda.current_device(), limit_all_gathers=True,
              sync_module_states=False)
    return fm, layer_cls


def lora_modules(m):
    return sum(1 for n, _ in m.named_parameters() if ".lora_A." in n and re.match(LORA_RE, n.split(".lora_A.")[0]))

### `trainkit/export.py`
Export: gather the LoRA weights one FSDP layer unit at a time -> PEFT adapter in the format vLLM served in p0.

In [ ]:
%%writefile /kaggle/working/trainkit/export.py
"""Export: gather the LoRA weights one FSDP layer unit at a time -> PEFT adapter in the format vLLM served in p0."""
import json
import os

import torch
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

from .config import TARGETS


def export_adapter(fm, layer_cls, out_dir, rank, r, alpha, base_path):
    """All ranks take part (each layer is one unit, ~0.3 GB gathered); rank 0 writes. Keys are
    base_model.model.<checkpoint module>.lora_{A,B}.weight, e.g. ...model.language_model.layers.N.self_attn.q_proj."""
    from safetensors.torch import save_file
    names = {id(mod): n for n, mod in fm.named_modules()}
    tensors = {}
    for unit in FSDP.fsdp_modules(fm):
        if not isinstance(unit.module, layer_cls):
            continue
        with FSDP.summon_full_params(unit, recurse=False, writeback=False):
            if rank == 0:
                base = names[id(unit)]
                for n, p in unit.module.named_parameters():
                    if "lora_" in n:
                        key = f"{base}.{n}".replace("_fsdp_wrapped_module.", "").removeprefix("m.").replace(".default.", ".")
                        tensors[key] = p.detach().to("cpu", torch.bfloat16).clone()
    if rank != 0:
        return None
    os.makedirs(out_dir, exist_ok=True)
    save_file(tensors, os.path.join(out_dir, "adapter_model.safetensors"))
    with open(os.path.join(out_dir, "adapter_config.json"), "w") as f:
        json.dump({"peft_type": "LORA", "task_type": "CAUSAL_LM", "r": r, "lora_alpha": alpha, "lora_dropout": 0.0,
                   "bias": "none", "fan_in_fan_out": False, "target_modules": list(TARGETS),
                   "base_model_name_or_path": base_path, "inference_mode": True}, f, indent=1)
    mods = {k.removeprefix("base_model.model.").rsplit(".lora_", 1)[0] for k in tensors}
    return {"dir": out_dir, "n_modules": len(mods), "n_tensors": len(tensors), "key_example": next(iter(tensors)),
            "shapes_ok": all(v.ndim == 2 and min(v.shape) == r for v in tensors.values()),
            "mb": round(os.path.getsize(os.path.join(out_dir, "adapter_model.safetensors")) / 1e6, 1)}

### `trainkit/worker.py`
Torchrun entry point (one process per GPU): python -m torch.distributed.run --nproc_per_node 4 -m trainkit.worker MODEL

In [ ]:
%%writefile /kaggle/working/trainkit/worker.py
"""
Torchrun entry point (one process per GPU): python -m torch.distributed.run --nproc_per_node 4 -m trainkit.worker MODEL
Data parallel: each rank runs one window per micro-step; FSDP averages gradients across ranks; AdamW steps every
grad_accum micro-steps with linear warmup then cosine decay. Validation loss (held-out tasks) at the start, every
val_every steps and at the end. Rank 0 writes metrics.jsonl and the adapter(s).
"""
import datetime
import json
import math
import sys
import time

import torch
import torch.distributed as dist

from .config import OUT_DIR, TrainSettings
from .data import TrainData
from .export import export_adapter
from .model import add_lora, load_base, lora_modules, wrap_fsdp


class Trainer:
    def __init__(self, s, model_path):
        self.s, self.model_path = s, model_path
        dist.init_process_group("gloo" if s.smoke else "nccl", timeout=datetime.timedelta(minutes=10))
        self.rank, self.world = dist.get_rank(), dist.get_world_size()
        if s.smoke:
            self.dev, self.dt = torch.device("cpu"), torch.float32
        else:
            torch.cuda.set_device(self.rank)
            self.dev, self.dt = torch.device("cuda", self.rank), torch.bfloat16
            # flash caps head_dim at 256 and math materialises T x T: force the memory-efficient kernel
            torch.backends.cuda.enable_flash_sdp(False)
            torch.backends.cuda.enable_math_sdp(False)
            torch.backends.cuda.enable_cudnn_sdp(False)
            torch.backends.cuda.enable_mem_efficient_sdp(True)
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        self.metrics = open(OUT_DIR / "metrics.jsonl", "a") if self.rank == 0 else None

    def log(self, **kw):
        kw["t_min"] = round((time.time() - self.s.t0) / 60, 1)
        print(f"rank{self.rank}", json.dumps(kw), flush=True)
        if self.metrics:
            self.metrics.write(json.dumps(kw) + "\n")
            self.metrics.flush()

    def build(self):
        t = time.time()
        m = load_base(self.model_path, self.rank, self.s.smoke)
        load_s = round(time.time() - t)
        m = add_lora(m, self.s.rank, self.s.alpha, self.dt)
        n_lora = lora_modules(m)
        self.fm, self.layer_cls = wrap_fsdp(m, self.s.smoke)
        if not self.s.smoke:
            torch.cuda.empty_cache()
        self.vocab = m.config.get_text_config().vocab_size
        self.log(event="model", load_s=load_s, lora_modules=n_lora,
                 gpu_gb=0 if self.s.smoke else round(torch.cuda.memory_allocated() / 2 ** 30, 2))

    def step_loss(self, data, w):
        ids, lab = data.example(w, self.dev)
        if ids.shape[1] >= self.s.offload_from:  # checkpoint inputs live in pinned host RAM (188 GB)
            with torch.autograd.graph.save_on_cpu(pin_memory=not self.s.smoke):
                return self.fm(ids, lab), int((lab != -100).sum())
        return self.fm(ids, lab), int((lab != -100).sum())

    def validate(self, data, tag):
        """Mean trained-token loss on the held-out tasks; every rank evaluates its share, results are reduced."""
        if not data.val_w:
            return None
        self.fm.eval()
        tot = torch.zeros(2, device=self.dev, dtype=torch.float64)
        mine = data.schedule(data.val_w, self.rank, self.world) or []
        with torch.no_grad():
            for w in mine:
                loss, n = self.step_loss(data, w)
                tot += torch.tensor([float(loss) * n, n], device=self.dev, dtype=torch.float64)
        dist.all_reduce(tot)
        self.fm.train()
        v = float(tot[0] / max(1.0, float(tot[1])))
        self.log(event="val", tag=tag, loss=round(v, 4), windows=len(mine) * self.world)
        return v

    def export(self, tag):
        out = OUT_DIR / ("adapter_" + tag)
        info = export_adapter(self.fm, self.layer_cls, str(out), self.rank, self.s.rank, self.s.alpha, self.model_path)
        dist.barrier()
        if info:
            self.log(event="export", tag=tag, **info)

    def lr_at(self, step, total):
        if step < self.s.warmup_steps:
            return self.s.lr * (step + 1) / self.s.warmup_steps
        p = (step - self.s.warmup_steps) / max(1, total - self.s.warmup_steps)
        return self.s.lr * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))

    def run(self, data):
        sched = data.schedule(data.train_w, self.rank, self.world, self.s.epochs)
        total = math.ceil(len(sched) / self.s.grad_accum)
        self.log(event="plan", micro_steps=len(sched), optimizer_steps=total, **data.stats())
        opt = torch.optim.AdamW([p for p in self.fm.parameters() if p.requires_grad], lr=self.s.lr, weight_decay=0.0)
        self.validate(data, "start")
        step, t_win, tok, loss_acc = 0, time.perf_counter(), 0, []
        for i, w in enumerate(sched):
            stop = torch.tensor([1.0 if self.s.hours_left() < 0.4 else 0.0], device=self.dev)
            dist.all_reduce(stop, op=dist.ReduceOp.MAX)  # every rank stops at the same micro-step
            if float(stop):
                self.log(event="budget_stop", micro_step=i, optimizer_step=step)
                break
            loss, n = self.step_loss(data, w)
            (loss / self.s.grad_accum).backward()
            loss_acc.append(float(loss.detach()))
            tok += data.length(w)
            if (i + 1) % self.s.grad_accum == 0 or i + 1 == len(sched):
                for g in opt.param_groups:
                    g["lr"] = self.lr_at(step, total)
                gn = self.fm.clip_grad_norm_(1.0)
                opt.step()
                opt.zero_grad(set_to_none=True)
                step += 1
                # Variable-length windows make world * rank0_tokens an underestimate.
                # Keep the actual workload and slowest rank's duration for auditing.
                if not self.s.smoke:
                    torch.cuda.synchronize(self.dev)  # include queued optimizer work in elapsed time
                dt = time.perf_counter() - t_win
                local_peak = 0 if self.s.smoke else torch.cuda.max_memory_allocated() / 2 ** 30
                tokens = torch.tensor([tok], device=self.dev, dtype=torch.int64)
                resources = torch.tensor([dt, local_peak], device=self.dev, dtype=torch.float64)
                dist.all_reduce(tokens, op=dist.ReduceOp.SUM)
                dist.all_reduce(resources, op=dist.ReduceOp.MAX)
                total_tokens, elapsed_s = int(tokens.item()), float(resources[0])
                self.log(event="step", step=step, of=total, loss=round(sum(loss_acc) / len(loss_acc), 4),
                         lr=round(self.lr_at(step - 1, total), 7), grad_norm=round(float(gn), 3),
                         tok_per_s=round(total_tokens / elapsed_s, 1),
                         tokens=total_tokens, elapsed_s=round(elapsed_s, 6),
                         rank_tokens=tok, rank_elapsed_s=round(dt, 6),
                         peak_gb=round(float(resources[1]), 2))
                t_win, tok, loss_acc = time.perf_counter(), 0, []
                if self.s.val_every and step % self.s.val_every == 0:
                    self.validate(data, f"step{step}")
                if self.s.save_every and step % self.s.save_every == 0:
                    self.export(f"step{step}")
        self.validate(data, "end")
        self.export("final")

    def close(self):
        dist.barrier()
        dist.destroy_process_group()


def main():
    s = TrainSettings.load()
    tr = Trainer(s, sys.argv[1])
    tr.build()
    data = TrainData(sys.argv[2], s.max_windows, vocab=tr.vocab if s.smoke else None, seed=s.seed)
    tr.run(data)
    tr.close()


if __name__ == "__main__":
    main()

### `trainkit/launcher.py`
Launcher: torchrun over the worker inside the scorer venv, with per-rank logs kept for post-mortems.

In [ ]:
%%writefile /kaggle/working/trainkit/launcher.py
"""Launcher: torchrun over the worker inside the scorer venv, with per-rank logs kept for post-mortems."""
import subprocess
import sys

from .config import OUT_DIR, WORK


def launch(model_path, data_path, nproc, timeout_s):
    logs = OUT_DIR / "rank_logs"
    logs.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(nproc), "--log-dir", str(logs),
           "--redirects", "3", "--tee", "3", "-m", "trainkit.worker", str(model_path), str(data_path)]
    print("$", " ".join(cmd), flush=True)
    env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "PYTHONPATH": str(WORK)}
    import os
    proc = subprocess.Popen(cmd, cwd=WORK, env={**os.environ, **env}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        return proc.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        proc.kill()
        return -9


def rank_tails(n=1500):
    """Last lines of each rank's stderr (a crashing rank's own traceback, which torchrun hides)."""
    out = {}
    for f in sorted((OUT_DIR / "rank_logs").glob("**/stderr.log")):
        t = f.read_text(errors="ignore")[-n:]
        if t.strip():
            out[str(f.relative_to(OUT_DIR))] = t
    return out

### `trainkit/main.py`
Everything after the env step, inside the scorer venv: python -m trainkit.main (cwd /kaggle/working).

In [ ]:
%%writefile /kaggle/working/trainkit/main.py
"""Everything after the env step, inside the scorer venv: python -m trainkit.main (cwd /kaggle/working)."""
import glob
import json
import shutil

from .config import DATA_FILE, INPUT, OUT_DIR, SMOKE_CONFIG, WORK, TrainSettings
from .launcher import launch, rank_tails


def model_path(smoke):
    if smoke:  # tiny random Gemma-4 from the real config (same architecture, 6 layers)
        import torch
        from transformers import AutoConfig, AutoModelForImageTextToText
        real = json.loads(SMOKE_CONFIG.read_text())
        tc, n = real["text_config"], 6
        tc.update(hidden_size=64, intermediate_size=128, num_hidden_layers=n, num_attention_heads=4,
                  num_key_value_heads=2, num_global_key_value_heads=1, head_dim=16, global_head_dim=32, vocab_size=512,
                  vocab_size_per_layer_input=512, sliding_window=16,
                  layer_types=["sliding_attention"] * (n - 1) + ["full_attention"])
        real["vision_config"].update(hidden_size=32, intermediate_size=64, num_hidden_layers=1, num_attention_heads=2,
                                     num_key_value_heads=2, head_dim=16, global_head_dim=16)
        torch.manual_seed(0)
        cfg = AutoConfig.for_model(**{k: v for k, v in real.items() if k != "transformers_version"})
        path = WORK / "tiny_model"
        AutoModelForImageTextToText.from_config(cfg).save_pretrained(path)
        return path
    cfgs = sorted(p for p in glob.glob(f"{INPUT}/**/config.json", recursive=True)
                  if "31b" in p.lower() and "unquantized" in p.lower())
    assert cfgs, "attach model google/gemma-4/Transformers/gemma-4-31b-it-qat-q4_0-unquantized/2"
    return cfgs[0].rsplit("/", 1)[0]


def data_path():
    if DATA_FILE.exists():
        return DATA_FILE
    found = sorted(glob.glob(f"{INPUT}/**/train_*.npz", recursive=True))
    assert found, "no training data: embed it (notebook inputs cell) or attach a dataset with train_*.npz"
    return found[0]


def main():
    s = TrainSettings.load()
    mp, dp = model_path(s.smoke), data_path()
    print("model", mp, "| data", dp, "| settings", json.dumps(s.__dict__), flush=True)
    rc = launch(mp, dp, 2 if s.smoke else 4, timeout_s=int(max(0.2, s.hours_left() + 0.3) * 3600))
    report = {"rc": rc, "settings": s.__dict__}
    mfile = OUT_DIR / "metrics.jsonl"
    rows = [json.loads(l) for l in open(mfile)] if mfile.exists() else []
    report["val"] = [r for r in rows if r.get("event") == "val"]
    report["exports"] = [r for r in rows if r.get("event") == "export"]
    steps = [r for r in rows if r.get("event") == "step"]
    report["steps"] = {"n": len(steps), "first": steps[:3], "last": steps[-3:]}
    if rc != 0:
        report["rank_tails"] = rank_tails()
    final = OUT_DIR / "adapter_final"
    if final.exists():  # ready to drop into a submission: adapters/<name>/
        dest = WORK / "adapters" / s.name
        shutil.rmtree(dest, ignore_errors=True)
        shutil.copytree(final, dest)
        report["adapter"] = str(dest)
    (WORK / "train_report.json").write_text(json.dumps(report, indent=1, default=str))
    print("TRAIN_REPORT", json.dumps({k: report[k] for k in ("rc", "val", "exports") if k in report}), flush=True)


if __name__ == "__main__":
    main()

## 5. Scorer environment
Restore the Py3.12 venv, upgrade changed wheels, add peft + bitsandbytes.

In [ ]:
sys.path.insert(0, "/kaggle/working")
from evalkit.envsetup import ScorerEnv
ScorerEnv().prepare_training(gpus=True)

## 6. Train
Runs inside the venv: torchrun over the GPUs (2 CPU ranks for the smoke); streamed here.

In [ ]:
import subprocess
proc = subprocess.Popen(["/tmp/scorer_host/bin/python", "-m", "trainkit.main"], cwd="/kaggle/working",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit code", proc.wait())

## 7. Results

In [ ]:
print(open("/kaggle/working/train_report.json").read()[:20000])